In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models


# --- 1. DEFINE YOUR PURE QUADRATIC LAYER ---
class PureQuadratic(nn.Module):
    def __init__(self, in_features, out_features):
        super(PureQuadratic, self).__init__()
        # ONLY one weight matrix (for the x^2 term) and a bias
        self.weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.bias = nn.Parameter(torch.Tensor(out_features))

        # Initialize weights (using Kaiming since it's the main weight now)
        nn.init.kaiming_uniform_(self.weight)
        nn.init.zeros_(self.bias)

    def forward(self, x):
        # Only the quadratic transformation: W * x^2 + b
        return torch.matmul(x**2, self.weight.t()) + self.bias


if __name__ == "__main__":
    # --- 2. SETUP DEVICE AND DATA ---
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    transform = transforms.Compose(
        [
            transforms.Resize(224),  # ResNet expects 224x224 images
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ]
    )

    # Download data
    trainset = torchvision.datasets.CIFAR10(
        root="./data", train=True, download=True, transform=transform
    )
    trainloader = torch.utils.data.DataLoader(
        trainset, batch_size=64, shuffle=True, num_workers=2
    )

    # --- 3. MODEL SETUP (TRANSFER LEARNING) ---
    # Load pre-trained ResNet18
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # Freeze all existing layers so they don't update during training
    for param in model.parameters():
        param.requires_grad = False

    # Replace the final fully connected layer with our PureQuadratic layer
    num_ftrs = model.fc.in_features
    model.fc = PureQuadratic(num_ftrs, 10)  # 10 classes for CIFAR-10

    # Move the model to the GPU (if available)
    model = model.to(device)

    # --- 4. OPTIMIZATION AND TRAINING ---
    criterion = nn.CrossEntropyLoss()

    # IMPORTANT: Only pass the parameters of the NEW layer to the optimizer
    optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

    print("Starting training...")
    model.train()

    for epoch in range(2):  # Start with 2 epochs to verify it works
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(trainloader):
            # Move data to GPU
            inputs, labels = inputs.to(device), labels.to(device)

            # Reset gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass and optimize
            loss.backward()
            optimizer.step()

            # Print statistics every 100 batches
            if i % 100 == 99:
                print(f"[Epoch {epoch + 1}, Batch {i + 1}] loss: {loss.item():.3f}")

    print("Training finished!")

    # Save the trained weights for the new pure quadratic model
    torch.save(model.state_dict(), "resnet_pure_quadratic.pth")

Using device: cpu
Files already downloaded and verified
Starting training...
[Epoch 1, Batch 100] loss: 1.657
[Epoch 1, Batch 200] loss: 0.829
[Epoch 1, Batch 300] loss: 0.902
[Epoch 1, Batch 400] loss: 0.933
[Epoch 1, Batch 500] loss: 0.935
[Epoch 1, Batch 600] loss: 0.801
[Epoch 1, Batch 700] loss: 0.971
[Epoch 2, Batch 100] loss: 0.843
[Epoch 2, Batch 200] loss: 0.757
[Epoch 2, Batch 300] loss: 0.862
[Epoch 2, Batch 400] loss: 0.600
[Epoch 2, Batch 500] loss: 0.532
[Epoch 2, Batch 600] loss: 0.451
[Epoch 2, Batch 700] loss: 0.844
Training finished!
